<a href="https://colab.research.google.com/github/dilbal/db125msc26project/blob/main/Phase2_SecondUniverse_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: Second-universe comparison

* Compare PCA-HRP with SML-ERC-HRP on ten alternative equity constituents. Use the first 79 test windows ending 3 June 2026, matching the main Phase 2 evaluation period. Training and test windows contain 252 and 63 observations.

* Inference uses the same four-window circular block bootstrap, null-centred two-sided test and 95% basic interval as main Phase 2. Two- and eight-window sensitivity results are also reported. This aligns the protocol; it does not establish statistical independence between universes or remove model-selection bias.

* Run top-to-bottom on a fresh runtime. If the fits are already in memory, run Section 11 alone to reuse their first 79 chunks.


## 1. Setup

Install dependencies and import shared libraries. The global NumPy seed and per-window optimiser seeds control different random-number generators. Versions are not pinned.


In [ ]:
!pip install yfinance scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from sklearn.neural_network import MLPRegressor
import yfinance as yf

np.random.seed(0)

## 2. Shared HRP allocator

* Single-linkage clustering determines an asset order.
* Midpoint recursive bisection allocates using inverse-variance cluster risk and raw training-return covariance.
* `SYMBOLS` is assigned to the second universe before the allocator is called.


In [ ]:
def quasi_diag(link, num_items):
    """Recursively sort clustered items into quasi-diagonal order."""
    link = link.astype(int)
    sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
    num_items = link[-1, 3]
    while sort_ix.max() >= num_items:
        sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
        df0 = sort_ix[sort_ix >= num_items]
        i = df0.index
        j = df0.values - num_items
        sort_ix[i] = link[j, 0]
        df0 = pd.Series(link[j, 1], index=i + 1)
        sort_ix = pd.concat([sort_ix, df0]).sort_index()
        sort_ix.index = range(sort_ix.shape[0])
    return sort_ix.tolist()

def cluster_var(cov_slice):
    """Inverse-variance portfolio variance for a cluster."""
    ivp = 1.0 / np.diag(cov_slice.values)
    ivp /= ivp.sum()
    return ivp @ cov_slice.values @ ivp

def get_rec_bipart(cov, sort_ix):
    """Recursive bisection: allocate weight inversely proportional to cluster variance."""
    w = pd.Series(1.0, index=sort_ix)
    c_items = [sort_ix]
    while len(c_items) > 0:
        c_items = [i[j:k] for i in c_items
                   for j, k in ((0, len(i) // 2), (len(i) // 2, len(i)))
                   if len(i) > 1]
        for i in range(0, len(c_items), 2):
            c0, c1 = c_items[i], c_items[i + 1]
            w0 = cluster_var(cov.loc[c0, c0])
            w1 = cluster_var(cov.loc[c1, c1])
            alpha = 1 - w0 / (w0 + w1)
            w[c0] *= alpha
            w[c1] *= 1 - alpha
    return w

def hrp_weights_from_dist(returns_df, dist_df):
    """Any asset-by-asset distance matrix in, HRP portfolio weights out.
    HRP's allocation step (clustering + recursive bisection) is identical
    regardless of which distance produced dist_df."""
    cov = returns_df.cov()
    condensed = squareform(dist_df.values, checks=False)
    link = linkage(condensed, method='single')
    sort_ix_pos = quasi_diag(link, len(returns_df.columns))
    sort_ix = [returns_df.columns[i] for i in sort_ix_pos]
    w = get_rec_bipart(cov, sort_ix)
    return (w / w.sum()).reindex(SYMBOLS)

def get_corr_distance(returns_df):
    """Pearson-correlation distance; retained as an unused reference helper."""
    corr = returns_df.corr()
    return np.sqrt((1 - corr) / 2)

print("HRP harness defined.")

## 3. Metrics and paired inference

* Sharpe uses a five-lag Bartlett-weighted adjustment with a variance floor.
* Maximum drawdown includes initial wealth 1; CEQ uses annualised arithmetic mean and sample variance.

* The paired bootstrap resamples circular blocks of four consecutive window differences. It uses 5,000 draws, seed 42, a null-centred absolute-tail test with a finite-simulation correction, and a 95% basic interval.


In [ ]:
def sharpe_lo2002(returns_series, periods=252, n_lags=5):
    """Five-lag Bartlett-weighted Sharpe adjustment with a variance floor."""
    mu, var = returns_series.mean(), returns_series.var()
    adj = 0.0
    for k in range(1, n_lags + 1):
        rho = returns_series.autocorr(lag=k)
        if not np.isnan(rho):
            adj += (1 - k / (n_lags + 1)) * rho
    return (mu / np.sqrt(max(var * (1 + 2 * adj), 1e-12))) * np.sqrt(periods)

def max_drawdown(r):
    # Include initial wealth so an immediate first-day loss is counted.
    cum = pd.concat([pd.Series([1.0]), (1 + r).cumprod()], ignore_index=True)
    peak = cum.cummax()
    return ((cum - peak) / peak).min()

def ceq(r, gamma, periods=252):
    """Certainty-equivalent return, annualised (DeMiguel et al. 2009 convention)."""
    return r.mean() * periods - (gamma / 2) * r.var() * periods

def quarterly_sharpes(chunks):
    return np.array([sharpe_lo2002(c) for c in chunks])

def bootstrap_sharpe_diff(chunks_a, chunks_b, n_boot=5000, seed=42, block_length=4):
    """Circular block bootstrap of paired window Sharpes; null-centred two-sided test.

    Default block length is four quarterly windows, with sensitivity reported below.
    This preserves local dependence within sampled blocks, not all dependence.
    Short regime samples remain exploratory. CI is a basic bootstrap interval.
    """
    if len(chunks_a) != len(chunks_b) or len(chunks_a) < 2:
        raise ValueError('Need at least two matching windows.')
    for a, b in zip(chunks_a, chunks_b):
        if not a.index.equals(b.index):
            raise ValueError('Paired return dates differ.')
    diffs = quarterly_sharpes(chunks_a) - quarterly_sharpes(chunks_b)
    if not np.isfinite(diffs).all():
        raise ValueError('Non-finite window Sharpe; inspect returns before inference.')
    n = len(diffs)
    length = int(block_length)
    if not 1 <= length <= n or n_boot < 100:
        raise ValueError('Invalid block length or bootstrap count.')
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, n, size=(n_boot, int(np.ceil(n / length))))
    indices = ((starts[..., None] + np.arange(length)) % n).reshape(n_boot, -1)[:, :n]
    obs = float(diffs.mean())
    null_means = (diffs - obs)[indices].mean(axis=1)
    q_lo, q_hi = np.quantile(null_means, [0.025, 0.975])
    p_value = (1 + np.count_nonzero(np.abs(null_means) >= abs(obs))) / (n_boot + 1)
    return obs, float(obs - q_hi), float(obs - q_lo), float(p_value)

print("Metrics and significance test defined.")

## 4. Asset features and weighted distance

Six features are computed from each training window and standardised across assets. `downside_dev` is the sample standard deviation of negative returns, not the zero-target downside deviation used in a Sortino ratio. Nonnegative feature weights define a diagonal Mahalanobis-style distance.


In [ ]:
from scipy.optimize import minimize

FEATURE_NAMES = ['mean_ret', 'vol', 'skew', 'kurt', 'avg_corr', 'downside_dev']

def build_asset_features(train_returns):
    corr = train_returns.corr()
    feats = {}
    for col in train_returns.columns:
        s = train_returns[col]
        mean_ret = s.mean() * 252
        vol = s.std() * np.sqrt(252)
        skew = s.skew()
        kurt = s.kurt()
        avg_corr = (corr[col].sum() - 1) / (len(corr.columns) - 1)
        downside = s[s < 0]
        downside_dev = downside.std() * np.sqrt(252) if len(downside) > 1 else 0.0
        feats[col] = [mean_ret, vol, skew, kurt, avg_corr, downside_dev]
    F = pd.DataFrame(feats, index=FEATURE_NAMES).T
    F = (F - F.mean()) / (F.std() + 1e-12)
    return F.copy()

def weighted_feature_distance(F, w):
    w = np.asarray(w)
    Fv = F.values
    n = len(Fv)
    Dm = np.zeros((n, n))
    for i in range(n):
        diff = (Fv - Fv[i])**2
        Dm[i] = np.sqrt((diff * w).sum(axis=1))
    np.fill_diagonal(Dm, 0.0)
    return pd.DataFrame(Dm, index=F.index, columns=F.index)

### 4.1 Retained variance-objective helpers

These reference helpers are retained from the original implementation.


In [ ]:
def insample_variance_objective(theta, F, train_returns):
    w = theta**2
    if w.sum() < 1e-8:
        return 1e6
    dist = weighted_feature_distance(F, w)
    try:
        wgt = hrp_weights_from_dist(train_returns, dist)
    except Exception:
        return 1e6
    cov = train_returns.cov()
    var = wgt.values @ cov.values @ wgt.values
    return float(var)

def fit_metric_weights(train_returns, n_restarts=4, maxiter=60, seed=0):
    feats_df = build_asset_features(train_returns)
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        theta0 = rng.uniform(0.1, 3.0, size=len(FEATURE_NAMES))
        res = minimize(insample_variance_objective, theta0, args=(feats_df, train_returns),
                        method='Nelder-Mead',
                        options={'maxiter': maxiter, 'xatol': 1e-2, 'fatol': 1e-9, 'adaptive': True})
        w = res.x**2
        if w.sum() > 1e-8 and (best is None or res.fun < best[0]):
            best = (res.fun, w)
    if best is None:
        return feats_df, np.ones(len(FEATURE_NAMES))
    return feats_df, best[1]

print('Supervised metric learning functions defined.')

## 5. PCA distance

* Standardise training returns, fit three components, and measure Euclidean distance between rows of `components_.T`. These coordinates are not multiplied by explained-variance scales.


In [ ]:
from sklearn.decomposition import PCA

def get_pca_distance(train_returns, n_components=3):
    X = train_returns.values
    X_std = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    pca = PCA(n_components=n_components)
    pca.fit(X_std)
    loadings = pca.components_.T
    n = loadings.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        diff = loadings - loadings[i]
        D[i] = np.sqrt((diff**2).sum(axis=1))
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, index=train_returns.columns, columns=train_returns.columns)

print("PCA distance (Phase 1 champion) defined.")

## 6. ERC objective and fitting

* Square the parameters to obtain nonnegative feature weights.
* HRP supplies the portfolio; the loss penalises deviations of relative asset variance contributions from 1/n.
* Four Nelder–Mead restarts have 60 iterations each.
* The lowest admissible achieved objective is retained without requiring reported convergence; equal feature weights are the fallback. Exact equal risk contributions or global optimality are not guaranteed.


In [ ]:
# Fit feature-distance weights with an equal-risk-contribution-style loss.
def erc_objective(theta, F, train_returns):
    w = theta**2
    if w.sum() < 1e-8:
        return 1e6
    dist = weighted_feature_distance(F, w)
    try:
        wgt = hrp_weights_from_dist(train_returns, dist)
    except Exception:
        return 1e6
    cov = train_returns.cov()
    wv = wgt.values
    port_var = wv @ cov.values @ wv
    if port_var < 1e-12:
        return 1e6
    marginal = cov.values @ wv
    rc = wv * marginal / port_var
    n = len(wv)
    return float(np.sum((rc - 1.0/n)**2))

def fit_metric_weights_erc(train_returns, n_restarts=4, maxiter=60, seed=0):
    feats_df = build_asset_features(train_returns)
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        theta0 = rng.uniform(0.1, 3.0, size=len(FEATURE_NAMES))
        res = minimize(erc_objective, theta0, args=(feats_df, train_returns), method='Nelder-Mead', options={'maxiter': maxiter, 'xatol': 1e-2, 'fatol': 1e-9, 'adaptive': True})
        w = res.x**2
        if w.sum() > 1e-8 and (best is None or res.fun < best[0]):
            best = (res.fun, w)
    if best is None:
        return feats_df, np.ones(len(FEATURE_NAMES))
    return feats_df, best[1]

print('SML-ERC functions defined.')

## 7. Universe and data-quality checks

* Alternative constituents: CVX, APD, CAT, HD, KO, UNH, BAC, AAPL, VZ and DUK. * Inspect missing observations before forward-filling.
* The diagnostic span ends on 3 June 2026, whereas the unrestricted download can extend later.


In [ ]:
# --- Second-universe robustness check: data-quality verification (run BEFORE the full 79-window pipeline) ---
TICKERS_2 = {
    'Energy': 'CVX', 'Materials': 'APD', 'Industrials': 'CAT',
    'Consumer Discretionary': 'HD', 'Consumer Staples': 'KO',
    'Healthcare': 'UNH', 'Financials': 'BAC',
    'Information Technology': 'AAPL', 'Communication Services': 'VZ',
    'Utilities': 'DUK',
}
SYMBOLS_2 = list(TICKERS_2.values())
print(f"Second universe: {SYMBOLS_2}")

raw_2 = yf.download(SYMBOLS_2, start='2004-01-01', auto_adjust=True)['Close']
raw_2 = raw_2[SYMBOLS_2]
span_start, span_end = pd.Timestamp('2005-08-19'), pd.Timestamp('2026-06-03')
window_check = raw_2.loc[span_start:span_end]
print("Rows in target span:", len(window_check))
print("\nPer-ticker diagnostics over target span (2005-08-19 to 2026-06-03):")
problem_tickers = []
for t in SYMBOLS_2:
    s = window_check[t]
    first_valid = s.first_valid_index()
    last_valid = s.last_valid_index()
    n_nan = s.isna().sum()
    if n_nan > 0:
        grp = s.isna().astype(int).groupby((~s.isna()).cumsum()).sum()
        max_gap = grp.max()
    else:
        max_gap = 0
    flag = ""
    if n_nan > 0 or first_valid is None or first_valid > span_start + pd.Timedelta(days=7):
        flag = "  <-- CHECK THIS TICKER"
        problem_tickers.append(t)
    print(f"  {t}: first_valid={first_valid.date() if first_valid is not None else None}, "
          f"last_valid={last_valid.date() if last_valid is not None else None}, "
          f"NaNs={n_nan}, max_consecutive_gap={max_gap}{flag}")

if problem_tickers:
    print(f"\nWARNING: tickers needing substitution: {problem_tickers}")
else:
    print("\nAll 10 tickers have clean, complete data across the full target span. No substitution needed.")

## 8. Returns and walk-forward windows

* Forward-fill prices and calculate complete simple daily returns.
* Build overlapping 252-observation training windows and nonoverlapping 63-observation tests, then keep the first 79 complete windows.
* Check that the final test ends on 3 June 2026. The unrestricted download does not freeze historical adjusted prices.


In [ ]:
# --- Second universe: build returns_2 and windows_2 (reusing the original walk-forward protocol) ---
TRAIN_DAYS = 252
TEST_DAYS = 63
FIRST_TRAIN_START = '2005-08-19'

raw_2 = raw_2.ffill()
returns_2 = raw_2.pct_change().dropna()
print(f"returns_2 shape: {returns_2.shape}")
assert returns_2.isnull().sum().sum() == 0, "NaN values found in returns_2"

def build_windows_2(returns, train_days, test_days, first_train_start):
    idx = returns.index
    start = idx.searchsorted(pd.Timestamp(first_train_start))
    windows, pos = [], start
    while pos + train_days + test_days <= len(idx):
        windows.append((idx[pos], idx[pos+train_days-1],
                        idx[pos+train_days], idx[pos+train_days+test_days-1]))
        pos += test_days
    return windows

windows_2 = build_windows_2(returns_2, TRAIN_DAYS, TEST_DAYS, FIRST_TRAIN_START)
windows_2 = windows_2[:79]
assert len(windows_2) == 79
assert windows_2[-1][3] == pd.Timestamp('2026-06-03')
print(f"Total windows (second universe): {len(windows_2)}")
print(f"First: {windows_2[0][0].date()} - {windows_2[0][3].date()}")
print(f"Last:  {windows_2[-1][0].date()} - {windows_2[-1][3].date()}")

## 9. PCA-HRP evaluation

* Fit PCA distances on each training sample and apply the resulting target weights to each daily test return.
* These are gross constant-weight returns.
* The turnover statistic is the full L1 change between successive target vectors


In [ ]:
# --- Second universe: PCA-HRP (the fixed champion) ---
# hrp_weights_from_dist() (defined earlier in this notebook) reindexes its output to the
# global SYMBOLS list; point it at the second universe for this section.
SYMBOLS = SYMBOLS_2

pca2_rets, pca2_chunks, pca2_turnover = [], [], []
prev_w_pca2 = None
for ts, te, vs, ve in windows_2:
    tr, tst = returns_2.loc[ts:te], returns_2.loc[vs:ve]
    dist = get_pca_distance(tr)
    w = hrp_weights_from_dist(tr, dist)
    r = (tst @ w[SYMBOLS_2]).rename('PCA-HRP-U2')
    pca2_rets.append(r); pca2_chunks.append(r)
    if prev_w_pca2 is not None:
        pca2_turnover.append(np.abs(w - prev_w_pca2).sum())
    prev_w_pca2 = w

pca2_series = pd.concat(pca2_rets)
print("=== PCA-HRP, second universe (robustness check) ===")
print(f"Sharpe (Lo 2002):       {sharpe_lo2002(pca2_series):.4f}")
print(f"Max Drawdown:           {max_drawdown(pca2_series):.4f}")
print(f"CEQ (gamma=1):          {ceq(pca2_series, 1):.4f}")
print(f"Avg quarterly turnover: {np.mean(pca2_turnover):.4f}")
print(f"N windows: {len(pca2_chunks)}")

## 10. SML-ERC-HRP evaluation

* Refit six feature weights in each training window using four optimiser restarts and seed k.
* Store fitted feature weights for later inspection.
* Gross returns and the target-change turnover statistic use the same conventions as PCA. Runtime increases with the number of complete windows.


In [ ]:
# --- Second universe: SML-ERC-HRP (the ERC-objective candidate) ---
# This is the expensive cell: all available windows x 4 Nelder-Mead restarts each fitting 6 feature weights
# against the Equal-Risk-Contribution loss. Expect this to take a while - that is expected.
erc2_rets, erc2_chunks, erc2_turnover, erc2_weights_log = [], [], [], []
prev_w_erc2 = None
for k, (ts, te, vs, ve) in enumerate(windows_2):
    tr, tst = returns_2.loc[ts:te], returns_2.loc[vs:ve]
    feats_df, w = fit_metric_weights_erc(tr, n_restarts=4, maxiter=60, seed=k)
    erc2_weights_log.append(w)
    dist = weighted_feature_distance(feats_df, w)
    wgt = hrp_weights_from_dist(tr, dist)
    r = (tst @ wgt[SYMBOLS_2]).rename('SML-ERC-HRP-U2')
    erc2_rets.append(r); erc2_chunks.append(r)
    if prev_w_erc2 is not None:
        erc2_turnover.append(np.abs(wgt - prev_w_erc2).sum())
    prev_w_erc2 = wgt
    if (k + 1) % 10 == 0:
        print(f"  ...window {k+1}/{len(windows_2)} done")

erc2_series = pd.concat(erc2_rets)
print("=== SML-ERC-HRP, second universe (robustness check) ===")
print(f"Sharpe (Lo 2002):       {sharpe_lo2002(erc2_series):.4f}")
print(f"Max Drawdown:           {max_drawdown(erc2_series):.4f}")
print(f"CEQ (gamma=1):          {ceq(erc2_series, 1):.4f}")
print(f"Avg quarterly turnover: {np.mean(erc2_turnover):.4f}")
print(f"N windows: {len(erc2_chunks)}")

## 11. Aligned comparison and block-length sensitivity

* Reuse the first 79 fitted chunks and verify their dates before calculating pooled gross metrics and paired inference.
* Positive differences favour ERC; the mean window difference is distinct from the difference in pooled Sharpes.
* The four-window result is the primary comparison. Results with block lengths two and eight assess sensitivity.
* No models are refitted by this cell. Earlier cleared outputs are superseded by this summary.


In [ ]:
def bootstrap_sharpe_diff(chunks_a, chunks_b, n_boot=5000, seed=42, block_length=4):
    """Circular block bootstrap of paired window Sharpes; null-centred two-sided test.

    Default block length is four quarterly windows, with sensitivity reported below.
    This preserves local dependence within sampled blocks, not all dependence.
    Short regime samples remain exploratory. CI is a basic bootstrap interval.
    """
    if len(chunks_a) != len(chunks_b) or len(chunks_a) < 2:
        raise ValueError('Need at least two matching windows.')
    for a, b in zip(chunks_a, chunks_b):
        if not a.index.equals(b.index):
            raise ValueError('Paired return dates differ.')
    diffs = quarterly_sharpes(chunks_a) - quarterly_sharpes(chunks_b)
    if not np.isfinite(diffs).all():
        raise ValueError('Non-finite window Sharpe; inspect returns before inference.')
    n = len(diffs)
    length = int(block_length)
    if not 1 <= length <= n or n_boot < 100:
        raise ValueError('Invalid block length or bootstrap count.')
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, n, size=(n_boot, int(np.ceil(n / length))))
    indices = ((starts[..., None] + np.arange(length)) % n).reshape(n_boot, -1)[:, :n]
    obs = float(diffs.mean())
    null_means = (diffs - obs)[indices].mean(axis=1)
    q_lo, q_hi = np.quantile(null_means, [0.025, 0.975])
    p_value = (1 + np.count_nonzero(np.abs(null_means) >= abs(obs))) / (n_boot + 1)
    return obs, float(obs - q_hi), float(obs - q_lo), float(p_value)

# Reuse completed fits: no optimisation or new data download.
assert len(windows_2) >= 79 and len(pca2_chunks) >= 79 and len(erc2_chunks) >= 79
windows_2 = windows_2[:79]
pca2_chunks = pca2_chunks[:79]
erc2_chunks = erc2_chunks[:79]
pca2_rets = pca2_chunks.copy()
erc2_rets = erc2_chunks.copy()
pca2_turnover = pca2_turnover[:78]
erc2_turnover = erc2_turnover[:78]
erc2_weights_log = erc2_weights_log[:79]
assert windows_2[-1][3] == pd.Timestamp('2026-06-03')
for (_, _, vs, ve), a, b in zip(windows_2, pca2_chunks, erc2_chunks):
    assert a.index.equals(b.index)
    assert len(a) == 63 and a.index[0] == vs and a.index[-1] == ve
pca2_series = pd.concat(pca2_chunks)
erc2_series = pd.concat(erc2_chunks)
assert len(pca2_series) == 4977 and pca2_series.index.is_unique
print('Aligned sample:', len(windows_2), 'windows;', pca2_series.index[0], 'to', pca2_series.index[-1])
summary_aligned = pd.DataFrame({
    'PCA-HRP': [sharpe_lo2002(pca2_series), max_drawdown(pca2_series), ceq(pca2_series,1), np.mean(pca2_turnover)],
    'SML-ERC-HRP': [sharpe_lo2002(erc2_series), max_drawdown(erc2_series), ceq(erc2_series,1), np.mean(erc2_turnover)]
}, index=['Pooled Sharpe','MaxDD','CEQ (gamma=1)','Mean target change']).T
print(summary_aligned.round(6).to_string())
rows = []
for length in (2,4,8):
    delta, low, high, p = bootstrap_sharpe_diff(erc2_chunks,pca2_chunks,block_length=length)
    rows.append([length,delta,low,high,p])
print(pd.DataFrame(rows,columns=['Block length','Delta ERC-PCA','CI low','CI high','p raw']).to_string(index=False))
obs2, lo2, hi2, p2 = bootstrap_sharpe_diff(erc2_chunks,pca2_chunks)
print('Primary four-window result:',obs2,lo2,hi2,p2)
